<a href="https://colab.research.google.com/github/yhuang-thu/FFmpeg/blob/master/convert_youtube_to_pptx.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
sudo apt-get update
sudo apt-get install tesseract-ocr
sudo apt-get install libtesseract-dev
pip install pytube
pip install pytesseract
pip install python-pptx
pip install reportlab


Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,628 B]
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://cli.github.com/packages stable InRelease [3,917 B]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [87.4 kB]
Get:9 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,933 kB]
Get:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [62.6 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [6,786

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 3.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


In [ ]:
from pytube import YouTube
import cv2
import pytesseract
from PIL import Image
from pptx import Presentation
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas
import os

In [ ]:
# Download YouTube video
video_url = "https://www.youtube.com/watch?v=VDc1Fjw6Lr8&list=PL8xZ-Bxw1Whs6_8XV--ht_3jGV7uMQHsy&index=13"
youtube = YouTube(video_url)
video_stream = youtube.streams.get_highest_resolution()
video_path = 'videos/video.mp4'
video_stream.download(output_path='videos', filename='video.mp4')

In [ ]:
# Create the 'frames' directory if it doesn't exist
os.makedirs('frames', exist_ok=True)

# Create a VideoCapture object
cap = cv2.VideoCapture(video_path)
frame_count = 0

# Create a PowerPoint presentation
ppt = Presentation()


while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
      break
    frame_count += 1
    image_path = f'frames/frame_{frame_count}.png'
    cv2.imwrite(image_path, frame)

    # Perform OCR on the image
    image = Image.open(image_path)
    text = pytesseract.image_to_string(image)

    # Create a new slide
    slide = ppt.slides.add_slide(ppt.slide_layouts[5])

    # Add image to the slide
    left = top = 0  # You might need to adjust these coordinates
    pic = slide.shapes.add_picture(image_path, left, top, height=ppt.slide_height)

    # Add extracted text to the slide
    left_text = 0  # You might need to adjust this coordinate
    top_text = pic.top + pic.height + 10  # Adjust the vertical distance from the image
    text_box = slide.shapes.add_textbox(left_text, top_text, pic.width, 100)
    text_frame = text_box.text_frame
    text_frame.text = text

cap.release()


In [ ]:
# Save the PowerPoint presentation
ppt.save('output.pptx')
ppt.save('output.pdf')
